# The Benefit of Data Sharing
Godwin et al. (2025) reviewed the open-science practices in recent **visual search** literature.<br>
They gracefully provided the dataset of articles they reviewed ([link](https://osf.io/5tmey/overview)), allowing others to explore the data further.<br>
Here, we analyze their dataset to examine whether sharing data is associated with increased citations. We rely on Godwin et al.'s classification of articles into four categories based on their data-sharing practices:
1. No data shared
2. Per-subject data
3. Per-trial data
4. Per-fixation data

We use the [OpenAlex](https://openalex.org/) API to retrieve citation counts for each of their articles and additional features of the publications (e.g. whether they were published with open access). We use these features, together with articles' data-sharing class, to predict their citations counts and [FWCI](https://help.openalex.org/hc/en-us/articles/24735753007895-Field-Weighted-Citation-Impact-FWCI) scores, and assess whether sharing data is associated with increased citations.

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from analysis.helpers.config import *
from analysis import duplicate_records_diagnostic
from analysis.helpers import dataset
from analysis.helpers.plotting import save_figure

pio.renderers.default = "browser"

In [2]:
combined, FEATURES_DF, CITATIONS_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (232 rows)


### Apply Exclusion Criteria
The Godwin et al. (2025) dataset contains over 500 records (of different articles). Our analysis focuses on a subset of articles that meet the following criteria:
1. A journal article - not a standalone OSF repo/project
2. Human eye-tracking data
3. Visual search related
4. Published within the Godwin et al. (2025) review period of 2017-2022 (though see caveat for 2016 articles below)
5. Has a valid DOI based on our OpenAlex query
6. Not retraced (based on the Godwin records)
7. not a duplicate of a different record (based on title-author or DOI match)

### Finalize Analytical Dataset
We combine the Godwin et al. (2025) data with the metadata we retrieved from OpenAlex to create our final analytical dataset.<br>
As a convension, `ALL_CAPS` column names are from the Godwin dataset; `CamelCase` column names are from OpenAlex metadata; and `snake_case` column names are derived or computed in this notebook.

In [3]:
FINAL_DATASET_SIZE = len(combined)
combined.columns

Index(['DOI', 'OpenAlexID', 'LastUpdate', 'PublicationType', 'PublicationYear',
       'PublicationDate', 'Topics', 'FieldWeightedCitationIndex',
       'IsRetracted', 'TotalCitations', 'Citations2025', 'Citations2024',
       'Citations2022', 'Citations2021', 'Citations2020', 'Error',
       'Citations2019', 'Citations2023', 'Citations2026', 'Citations2018',
       'Citations2017', 'Citations2016', 'Citations2014', 'Citations2013',
       'Citations2015', 'Pub2UpdateTime', 'Authors', 'NumAuthors',
       'HasUSAuthor', 'IsOpenAccess', 'HasPreprint', 'PreprintSources',
       'VenueID', 'VenueName', 'Venue2yrMeanCitedness', 'VenueHIndex',
       'VenueI10Index', 'PAPER_LINK', 'YEAR_PUBLISHED', 'PAPER_TITLE',
       'AUTHORS', 'PUBLISHED_IN_YEAR_RANGE', 'IS_PUBLICATION',
       'IS_PRIMARY_RESEARCH_HUMAN', 'IS_VISUAL_SEARCH', 'IS_EYE_TRACKING',
       'CLAIMED_TO_SHARE', 'SHARING_LINK', 'SHARING_LOCATION', 'OSF_ID',
       'FOUND_ATTEMPT_1', 'FOUND_ATTEMPT_2', 'ACTUALLY_SHARED', 'EXPERI

---

## Data Quality Checks
Before examining discrepancies between sources, we check two structural properties of the
analytic sample: whether any duplicate records slipped through deduplication, and whether
OpenAlex measured sharing and non-sharing articles at comparable times.

### Duplicate Records
Reviewer 1 asked us to track down the duplicate-DOI issue noted in the Method section.
`duplicate_records_diagnostic.find_duplicate_doi_records()` identifies the source rows that
share an OpenAlex-resolved DOI; we annotate here whether each one survived into the
analytic sample (only one row per duplicated DOI should - the others are the removable
duplicates).<br>
This report casts a wider net than the exclusion cascade below: it flags every duplicated DOI
in the full Godwin corpus, not only those among records that already passed the topic, year,
and retraction criteria. That is why it finds more duplicate pairs than the "6" figure in the
exclusion counts - most involve a record that would have been excluded on other grounds
anyway, regardless of the duplication.

In [ ]:
from analysis import duplicate_records_diagnostic

duplicate_report = duplicate_records_diagnostic.find_duplicate_doi_records(
    dataset_path=str(GODWIN_PATH), metadata_path=str(METADATA_PATH)
)
duplicate_report["in_analytic_sample"] = duplicate_report.index.isin(combined.index)

n_doi = duplicate_report["doi_norm"].nunique()
n_rows = len(duplicate_report)
n_conflicting = duplicate_report.groupby("doi_norm")["data_sharing_class"].nunique().gt(1).sum()
n_kept = duplicate_report["in_analytic_sample"].sum()
print(f"Duplicated DOIs: {n_doi}  (spanning {n_rows} source rows; "
      f"{n_rows - n_doi} rows are removable duplicates)")
print(f"Duplicate pairs with conflicting data_sharing_class: {n_conflicting}")
print(f"Duplicate rows retained in the analytic sample: {n_kept} "
      f"(expected: {n_doi}, one survivor per DOI)")

duplicate_report

### Census Times
OpenAlex records when it last checked each article's metadata (`LastUpdate`). If sharing and
non-sharing articles were checked at systematically different times, downstream citation
comparisons would be confounded by measurement time rather than sharing status alone.

In [ ]:
census_times = (
    combined
    .assign(share_label=combined["is_sharing_data"].map({True: "SHARING", False: "NOT SHARING"}))
    .groupby("share_label")["LastUpdate"]
    .agg(["count", "min", "median", "max"])
)
census_times

---

## Check Discrepancies
We examine some discrepancies between the Godwin et al. (2025) dataset and the OpenAlex metadata we retrieved.

### Publication-Year Discrepancies
Some records in the Godwin dataset have a different publication year than in OpenAlex metadata.<br>
We show here how many discrepancies there are, but do not exclude any articles based on this.

In [4]:
# the pre-exclusion frame, needed to see what each criterion removes
merged = dataset.build_merged()

Loaded OpenAlex metadata from CSV.


In [5]:
# exclude publications but disregard publication year criterion:
merged_subset = (
    merged
    .loc[  # correct topic
        merged[["IS_PRIMARY_RESEARCH_HUMAN", "IS_VISUAL_SEARCH", "IS_EYE_TRACKING"]].eq("YES").all(axis=1)
    ]
    .loc[merged["DOI"].notna()]  # successful query
    .loc[merged["IsRetracted"] == False]  # not retracted
    .loc[~merged.duplicated(subset="DOI", keep="first")]  # not dup DOI
)
print(f"Num of publications without excluding by publication year:\t {merged_subset.shape[0]}")

# calculate how many articles would be rejected by either database's publication year:
godwin_pass_year_range = merged_subset["YEAR_PUBLISHED"].between(2017, 2022).rename("Godwin")
openalex_pass_year_range = merged_subset["PublicationYear"].between(2017, 2022).rename("OpenAlex")
confusion = pd.crosstab(
    godwin_pass_year_range, openalex_pass_year_range, rownames=["Godwin"], colnames=["OpenAlex"],
)
display(confusion)

print("Publication-year discrepancy:")
display(
    merged_subset.loc[
        godwin_pass_year_range & ~openalex_pass_year_range, ["YEAR_PUBLISHED", "PublicationYear"]
    ].rename(columns={"YEAR_PUBLISHED": "Godwin_Year", "PublicationYear": "OpenAlex_Year"})
)

Num of publications without excluding by publication year:	 236


OpenAlex,False,True
Godwin,,
False,4,0
True,10,222


Publication-year discrepancy:


,Godwin_Year,OpenAlex_Year
49,2017.0,2016.0
78,2017.0,2016.0
114,2017.0,2016.0
116,2017.0,2016.0
163,2017.0,2016.0
198,2017.0,2016.0
203,2017.0,2016.0
206,2017.0,2016.0
212,2017.0,2016.0
247,2018.0,2016.0


### Author Discrepancies

In [6]:
print("### Missing Authorship ###")
godwin_missing_auths_count = combined["AUTHORS"].isna().sum()
print(f"Articles with missing authorship in the Godwin Dataset:\t {godwin_missing_auths_count} ({100 * godwin_missing_auths_count / FINAL_DATASET_SIZE :.1f}%)")
openalex_missing_auths_count = combined["Authors"].map(lambda auths: not auths or auths == "[]").sum()
print(f"Articles with missing authorship in the OpenAlex Dataset:\t {openalex_missing_auths_count} ({100 * openalex_missing_auths_count / FINAL_DATASET_SIZE :.1f}%)")

print ("### Authorship Discrepancy ###")
has_godwin_authors = (
    combined
    .loc[combined["AUTHORS"].notna(), ["DOI", "AUTHORS", "Authors", "NumAuthors"]]
    .rename(columns={
        "AUTHORS": "AUTHORS (Godwin)", "Authors": "Authors (OpenAlex)", "NumAuthors": "NumAuthors (OpenAlex)"
    })
)
has_godwin_authors["NUM_AUTHORS (Godwin)"] = has_godwin_authors["AUTHORS (Godwin)"].str.split("; ").str.len()
is_auth_count_bad = has_godwin_authors["NUM_AUTHORS (Godwin)"] != has_godwin_authors["NumAuthors (OpenAlex)"]
print(f"There are {is_auth_count_bad.sum()} ({100 * is_auth_count_bad.mean() :.1f}%) rows with mismatching author-counts")
has_godwin_authors.loc[is_auth_count_bad]

### Missing Authorship ###
Articles with missing authorship in the Godwin Dataset:	 14 (6.0%)
Articles with missing authorship in the OpenAlex Dataset:	 0 (0.0%)
### Authorship Discrepancy ###
There are 2 (0.9%) rows with mismatching author-counts


,DOI,AUTHORS (Godwin),Authors (OpenAlex),NumAuthors (OpenAlex),NUM_AUTHORS (Godwin)
186,https://doi.org/10.16910/jemr.10.2.6,"Van der Stigchel, S; Meeter, M","Van der Stigchel, S",1.0,2
205,https://doi.org/10.1167/18.13.5,"Perez, DL; Radkowska, A; Raczaszek-Leonardi, J...","Pérez, DL; Radkowska, A; Rączaszek‐Leonardi, J...",5.0,4
